In [29]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from uuid import UUID
import time_uuid
from datetime import datetime, timedelta
load_dotenv()

MYSQL_URL = f"jdbc:mysql://{os.getenv('MYSQL_HOST')}:{os.getenv('MYSQL_PORT')}/{os.getenv('MYSQL_DB')}"
MYSQL_HOST = os.getenv('MYSQL_HOST')
MYSQL_PORT = os.getenv('MYSQL_PORT')
MYSQL_DB = os.getenv('MYSQL_DB')
MYSQL_USER = os.getenv('MYSQL_USER')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD')
MYSQL_JAR = os.getenv('MYSQL_JAR')

CASSANDRA_HOST = os.getenv('CASSANDRA_HOST')
CASS_KEYSPACE = os.getenv('CASSANDRA_KEYSPACE')
CASS_TABLE = os.getenv('CASSANDRA_TABLE')
CASSANDRA_JAR = os.getenv('CASSANDRA_JAR')

spark = SparkSession.builder \
    .appName("Connect_Cassandra_MySQL") \
    .config("spark.jars", f"{MYSQL_JAR},{CASSANDRA_JAR}") \
    .config("spark.cassandra.connection.host", CASSANDRA_HOST) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .getOrCreate()

def read_data_table_from_cassandra(table, keyspace):
    df_cassandra = spark.read \
        .format("org.apache.spark.sql.cassandra") \
        .options(table=table, keyspace=keyspace) \
        .load()
    return df_cassandra

def import_to_mysql(output, mysql_url, mysql_user, password):
    final_output = output.select('job_id', 'date', 'hour', 'publisher_id', 'company_id', 'campaign_id', 'group_id',
                                 'unqualified', 'qualified', 'conversions', 'clicks', 'bid_set', 'spend_hour')
    final_output = final_output.withColumnRenamed('date', 'dates').withColumnRenamed('hour', 'hours').withColumnRenamed(
        'qualified', 'qualified_application'). \
        withColumnRenamed('unqualified', 'disqualified_application').withColumnRenamed('conversions', 'conversion')
    final_output = final_output.withColumn('sources', lit('Cassandra'))
    final_output.printSchema()
    final_output.write.format("jdbc") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .option("url", mysql_url) \
        .option("dbtable", "events") \
        .mode("append") \
        .option("user", mysql_user) \
        .option("password", password) \
        .save()
    return print('Data imported successfully')

def transform_data_from_cassandra(data):
    data = data.select('create_time','job_id','custom_track','bid','campaign_id','group_id','publisher_id')
    data = data.filter(data.job_id.isNotNull())
    data = data.filter(data.custom_track.isNotNull())
    return data

def get_latest_time_cassandra():
    data = spark.read.format("org.apache.spark.sql.cassandra").options(table="tracking", keyspace="recruitment").load()
    cassandra_lastest_time = data.agg({'ts' : 'max'}).take(1)[0][0]
    return cassandra_lastest_time

def get_mysql_lastest_time(url, driver, user, password):
    sql = """(select max(updated_at) from events) data"""
    mysql_time = spark.read.format('jdbc').options(url=url, driver=driver, dbtable=sql, user=user, password=password).load()
    mysql_time=mysql_time.take(1)[0][0]
    if mysql_time is None:
        mysql_latest = '1998-01-01 00:00:00'
    else:
        mysql_latest = mysql_time.strftime('%Y-%m-%d %H:%M:%S')
    return mysql_latest

def calculating_clicks(df):
    clicks_data = df.filter(df.custom_track == 'click')
    clicks_data = clicks_data.na.fill({'bid': 0})
    clicks_data = clicks_data.na.fill({'job_id': 0})
    clicks_data = clicks_data.na.fill({'publisher_id': 0})
    clicks_data = clicks_data.na.fill({'group_id': 0})
    clicks_data = clicks_data.na.fill({'campaign_id': 0})
    clicks_data.registerTempTable('clicks')
    clicks_output = spark.sql(
        """ select job_id , date(ts) as date , hour(ts) as hour ,
                publisher_id , campaign_id , group_id ,
                avg(bid) as bid_set, count(*) as clicks , sum(bid) as spend_hour
            from clicks
            group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id
        """)
    return clicks_output

def calculating_conversion(df):
    conversion_data = df.filter(df.custom_track == 'conversion')
    conversion_data = conversion_data.na.fill({'job_id': 0})
    conversion_data = conversion_data.na.fill({'publisher_id': 0})
    conversion_data = conversion_data.na.fill({'group_id': 0})
    conversion_data = conversion_data.na.fill({'campaign_id': 0})
    conversion_data.registerTempTable('conversion')
    conversion_output = spark.sql(
        """ select job_id, date (ts) as date, hour (ts) as hour,
              publisher_id, campaign_id, group_id,
                count (*) as conversions
            from conversion
            group by job_id, date (ts), hour (ts), publisher_id, campaign_id, group_id
        """)
    return conversion_output

def calculating_qualified(df):
    qualified_data = df.filter(df.custom_track == 'qualified')
    qualified_data = qualified_data.na.fill({'job_id': 0})
    qualified_data = qualified_data.na.fill({'publisher_id': 0})
    qualified_data = qualified_data.na.fill({'group_id': 0})
    qualified_data = qualified_data.na.fill({'campaign_id': 0})
    qualified_data.registerTempTable('qualified')
    qualified_output = spark.sql(
        """ select job_id , date(ts) as date , hour(ts) as hour ,
                publisher_id , campaign_id , group_id , count(*) as qualified
            from qualified
            group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id
        """)
    return qualified_output

def calculating_unqualified(df):
    unqualified_data = df.filter(df.custom_track == 'unqualified')
    unqualified_data = unqualified_data.na.fill({'job_id':0})
    unqualified_data = unqualified_data.na.fill({'publisher_id':0})
    unqualified_data = unqualified_data.na.fill({'group_id':0})
    unqualified_data = unqualified_data.na.fill({'campaign_id':0})
    unqualified_data.registerTempTable('unqualified')
    unqualified_output = spark.sql(
        """ select job_id , date(ts) as date , hour(ts) as hour ,
               publisher_id , campaign_id , group_id , count(*) as unqualified
            from unqualified
            group by job_id , date(ts) , hour(ts) , publisher_id , campaign_id , group_id
        """)
    return unqualified_output


def process_final_data(clicks_output, conversion_output, qualified_output, unqualified_output):
    final_data = clicks_output.join(conversion_output,
                                    ['job_id', 'date', 'hour', 'publisher_id', 'campaign_id', 'group_id'], 'full'). \
        join(qualified_output, ['job_id', 'date', 'hour', 'publisher_id', 'campaign_id', 'group_id'], 'full'). \
        join(unqualified_output, ['job_id', 'date', 'hour', 'publisher_id', 'campaign_id', 'group_id'], 'full')
    return final_data

def process_cassandra_data(df):
    clicks_output = calculating_clicks(df)
    clicks_output.show(10)
    conversion_output = calculating_conversion(df)
    conversion_output.show(10)
    qualified_output = calculating_qualified(df)
    qualified_output.show(10)
    unqualified_output = calculating_unqualified(df)
    unqualified_output.show(10)

def retrieve_company_data(url,driver,user,password):
    sql = """(SELECT id as job_id, company_id, group_id, campaign_id FROM job) test"""
    company = spark.read.format('jdbc').options(url=url, driver=driver, dbtable=sql, user=user, password=password).load()
    return company

# driver = 'com.mysql.cj.jdbc.Driver'
# print('The host is ' ,MYSQL_HOST)
# print('The port using is ',MYSQL_PORT)
# print('The db using is ',MYSQL_DB)
# print('-----------------------------')
# print('Retrieving data from Cassandra')
# print('-----------------------------')
# # df = spark.read.format("org.apache.spark.sql.cassandra").options(table="tracking", keyspace="recruitment").load().where(col('ts') >= mysql_time)
# df = read_data_table_from_cassandra('tracking', 'recruitment')
# # print('-----------------------------')
# # print('Selecting data from Cassandra')
# # print('-----------------------------')
# # df = df.select('ts', 'job_id', 'custom_track', 'bid', 'campaign_id', 'group_id', 'publisher_id')
# # df = df.filter(df.job_id.isNotNull())
# # df.printSchema()
# # df.show(10)
# print('-----------------------------')
# print('Processing Cassandra Output')
# print('-----------------------------')
# df = df.filter(df.job_id.isNotNull())
# cassandra_output = process_cassandra_data(df)
# cassandra_output.filter(cassandra_output.conversions.isNotNull()).show(10)
# # print('-----------------------------')
# # print('Merge Company Data')
# # print('-----------------------------')
# # company = retrieve_company_data(MYSQL_URL, driver , MYSQL_USER, MYSQL_PASSWORD)
# # print('-----------------------------')
# # print('Finalizing Output')
# # print('-----------------------------')
# # final_output = cassandra_output.join(company, 'job_id', 'left').drop(company.group_id).drop(company.campaign_id)
# # final_output.show(10)
# # print('-----------------------------')
# # print('Import Output to MySQL')
# # print('-----------------------------')
# # import_to_mysql(final_output, MYSQL_URL, MYSQL_USER, MYSQL_PASSWORD)
#
# # df1 = calculating_clicks(df)
# # df1.show(5)
# # df2 = calculating_conversion(df)
# # df2.show(5)
# # df3 = calculating_qualified(df)
# # df3.show(5)
# # df4 = calculating_unqualified(df)
# # df4.show(5)

In [35]:
driver = 'com.mysql.cj.jdbc.Driver'
print('The host is ' ,MYSQL_HOST)
print('The port using is ',MYSQL_PORT)
print('The db using is ',MYSQL_DB)
print('-----------------------------')
print('Retrieving data from Cassandra')
print('-----------------------------')
df = read_data_table_from_cassandra('tracking', 'recruitment')
# print('-----------------------------')
# print('Processing Cassandra Output')
# print('-----------------------------')
# df = df.filter(df.job_id.isNotNull())
# process_cassandra_data(df)
# cassandra_output.filter(cassandra_output.conversions.isNotNull()).show(10)
# # print('-----------------------------')
# # print('Merge Company Data')
# # print('-----------------------------')
# # company = retrieve_company_data(MYSQL_URL, driver , MYSQL_USER, MYSQL_PASSWORD)
# # print('-----------------------------')
# # print('Finalizing Output')
# # print('-----------------------------')
# # final_output = cassandra_output.join(company, 'job_id', 'left').drop(company.group_id).drop(company.campaign_id)
# # final_output.show(10)
# # print('-----------------------------')
# # print('Import Output to MySQL')
# # print('-----------------------------')
# # import_to_mysql(final_output, MYSQL_URL, MYSQL_USER, MYSQL_PASSWORD)
#
# # df1 = calculating_clicks(df)
# # df1.show(5)
# # df2 = calculating_conversion(df)
# # df2.show(5)
# # df3 = calculating_qualified(df)
# # df3.show(5)
# # df4 = calculating_unqualified(df)
# # df4.show(5)

The host is  mysql
The port using is  3306
The db using is  etl_db
-----------------------------
Retrieving data from Cassandra
-----------------------------


In [39]:
process_cassandra_data(df)
df = df.filter(df.job_id.isNotNull()).filter(df.custom_track == 'qualified')
df.show(10)

+------+----------+----+------------+-----------+--------+------------------+------+------------------+
|job_id|      date|hour|publisher_id|campaign_id|group_id|           bid_set|clicks|        spend_hour|
+------+----------+----+------------+-----------+--------+------------------+------+------------------+
|    98|2022-07-26|   6|           1|          4|       0|               2.0|    14|              28.0|
|   187|2022-07-27|   4|           1|         48|      34|               1.5|    15|              22.5|
|  1530|2022-07-25|   9|           1|        222|       0|               0.0|    11|               0.0|
|   273|2022-07-26|   6|           1|         48|       0|               0.0|    11|               0.0|
|   188|2022-07-24|  14|           1|         48|      34|               1.0|    86|              86.0|
|  1530|2022-07-27|   4|           1|        222|       0|               0.0|    10|               0.0|
|   336|2022-07-26|   2|           1|         15|       0|1.3999